In [3]:
!pip install unsloth
# Also get the latest trl and peft
!pip install --upgrade trl peft

  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.29.0
    Uninstalling trl-0.29.0:
      Successfully uninstalled trl-0.29.0
  Using cached trl-0.29.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.29.0-py3-none-any.whl (528 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.3.3 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 0.29.0 which is incompatible.
unsloth-zoo 2026.3.1 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 0.29.0 which is incompatible.


In [4]:
from google.colab import userdata
from huggingface_hub import login
import wandb

# Load secrets from Colab
hf_token = userdata.get('HF_TOKEN')
wandb_token = userdata.get('wnb')

# Log in
login(token=hf_token)
wandb.login(key=wandb_token)
wandb.init(project="medical-qlora-unsloth", job_type="training")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: huzaifa_as (huzaifa_as-bahria-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
# 1. Install modelscope
!pip install modelscope

# 2. Tell Unsloth to use ModelScope instead of Hugging Face
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

# 3. Load the model as usual
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token, # You can still pass your HF token here
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 94.4 MB/s eta 0:00:00
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


2026-03-06 21:36:46,215 - modelscope - INFO - Got 9 files, start to download ...


Processing 9 items:   0%|          | 0.00/9.00 [00:00<?, ?it/s]

2026-03-06 21:40:26,810 - modelscope - INFO - Download model 'unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit' successfully.


==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /root/.cache/modelscope/hub/models/unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit as a legacy tokenizer.


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: Controls the size of the adapter (16 is a solid default)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Kept at 0 for optimized Unsloth training
    bias = "none",    # Kept at "none" for optimized Unsloth training
    use_gradient_checkpointing = "unsloth", # Crucial for saving VRAM on long contexts!
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# Let's check how many parameters we are actually training
model.print_trainable_parameters()

Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [7]:
from datasets import load_dataset

# Added 'en' to specify the English configuration of the dataset
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split="train[:500]")

# Define the prompt template
train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
### Instruction:
{}
### Question:
{}
### Response:
<think>
{}
</think>
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    # Using .get() to safely pull from the dictionary regardless of exact casing
    inputs  = examples.get("Question", examples.get("question", []))
    cots    = examples.get("Complex_CoT", examples.get("complex_cot", []))
    outputs = examples.get("Response", examples.get("response", []))

    texts = []

    # Loop through the batch
    for input, cot, output in zip(inputs, cots, outputs):
        instruction = "Answer the following medical question with step-by-step reasoning."

        # Format the text and append the EOS token
        text = train_prompt_style.format(instruction, input, cot, output) + EOS_TOKEN
        texts.append(text)

    return { "text" : texts }

# Apply the formatting
dataset = dataset.map(formatting_prompts_func, batched = True)

# --- SANITY CHECK ---
print("✅ Dataset loaded successfully. Here is what Example 0 looks like:\n")
print("-" * 50)
print(dataset[0]["text"])
print("-" * 50)

README.md: 0.00B [00:00, ?B/s]

medical_o1_sft.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

✅ Dataset loaded successfully. Here is what Example 0 looks like:

--------------------------------------------------
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
### Instruction:
Answer the following medical question with step-by-step reasoning.
### Question:
Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?
### Response:
<think>
Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?

But wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lo

In [8]:
from trl import SFTTrainer, SFTConfig # CHANGED: Imported SFTConfig instead of TrainingArguments
import torch

# 1. Define the Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = SFTConfig( # CHANGED: Using SFTConfig here
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "medical_outputs",
        report_to = "wandb",
    ),
)

# 2. Check Memory BEFORE Training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.\n")

# 3. START TRAINING!
trainer_stats = trainer.train()

# 4. Check Memory AFTER Training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"\n{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
GPU = Tesla T4. Max memory = 14.563 GB.
5.787 GB of memory reserved.



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 63
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,1.941056
20,1.503775
30,1.485691
40,1.441163
50,1.471448
60,1.446920


train/epoch,▁▂▄▅▆██
train/global_step,▁▂▄▅▆██
train/grad_norm,█▂▂▁▂▁
train/learning_rate,█▇▅▄▂▁
train/loss,█▂▂▁▁▁
total_flos,1.4980517314584576e+16
train/epoch,1
train/global_step,63
train/grad_norm,0.26679
train/learning_rate,1e-05
train/loss,1.44692



1075.2563 seconds used for training.
17.92 minutes used for training.
Peak reserved memory = 7.594 GB.
Peak reserved memory for training = 1.807 GB.
Peak reserved memory % of max memory = 52.146 %.
Peak reserved memory for training % of max memory = 12.408 %.


In [9]:
import os

adapter_save_path = "medical_qlora_adapter"

# Save the adapter and tokenizer locally
model.save_pretrained(adapter_save_path)
tokenizer.save_pretrained(adapter_save_path)

print(f"✅ Adapter successfully saved to folder: '{adapter_save_path}'")
print("Files saved:", os.listdir(adapter_save_path))


✅ Adapter successfully saved to folder: 'medical_qlora_adapter'
Files saved: ['tokenizer_config.json', 'README.md', 'tokenizer.json', 'chat_template.jinja', 'adapter_config.json', 'adapter_model.safetensors']


In [10]:
from unsloth import FastLanguageModel

# Enable native 2x faster inference in Unsloth
FastLanguageModel.for_inference(model)

# Define a brand new medical scenario
test_instruction = "Answer the following medical question with step-by-step reasoning."
test_question = "A 65-year-old male presents to the emergency department with sudden onset of right-sided weakness and difficulty speaking. His symptoms started 45 minutes ago. His medical history is significant for atrial fibrillation, but he stopped taking his blood thinners a month ago. What is the most likely diagnosis, and what is the immediate next step in management?"

# Format the prompt using our template (leaving the reasoning and response parts empty for the model to fill)
inputs = tokenizer(
    [train_prompt_style.format(test_instruction, test_question, "", "")],
    return_tensors = "pt"
).to("cuda")

# Generate the output
outputs = model.generate(
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 600, # Give it enough room to think and answer
    use_cache = True
)

# Decode the generated tokens back into text
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# Print just the generated response section for clarity
print("\n" + "="*50)
print("🤖 MEDICAL MODEL OUTPUT")
print("="*50)
try:
    print(response.split("### Response:\n")[1])
except IndexError:
    print(response) # Fallback just in case the split doesn't find the exact string

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 


🤖 MEDICAL MODEL OUTPUT
<think>

</think>
Alright, let's think about this. This guy is 65, and he's suddenly having trouble speaking and moving his right side. That sounds like a stroke to me. It's like a sudden onset of weakness on one side, and it's been 45 minutes. That's pretty quick. He's got a history of atrial fibrillation, but he stopped his blood thinners a month ago. Hmm, that's interesting.

Okay, atrial fibrillation usually means he's at risk for blood clots, especially in the brain. If he's not on his blood thinners, there's a higher chance of a clot forming somewhere. And when those clots go to the brain, they can cause a stroke. That's kind of what's happening here.

So, given his symptoms and history, this is looking like a possible stroke caused by a blood clot. Now, we need to figure out what's going on with the clot. Is it in the carotid artery, or maybe somewhere else like the heart? The carotid artery is a common culprit for these kinds of strokes.

To really pinpo